# Supporting Tables

This notebook generates the Supporting Information tables for the hydrovoltaic-ML manuscript.

Tables generated:
- Table S1. Full data dictionary and descriptor encoding
- Table S2. Dataset scope, inclusion criteria, and excluded system types
- Table S3. Dataset summary and reporting completeness
- Table S4. Model hyperparameters and full performance metrics
- Table S5. Descriptor augmentation and feature-block ablation results
- Table S6. SHAP descriptor importance ranking
- Table S7. Internal-resistance group statistics and statistical tests
- Table S8. Explicit descriptor model coefficients and virtual design candidates

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
SI_DIR = RESULTS_DIR / "supporting_materials"
TABLE_DIR = SI_DIR / "tables"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Supporting table folder:", TABLE_DIR)
print("Generated:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Project root: C:\Users\wenlu\OneDrive\books_coding\Python_Projects\hydrovoltaic-ml
Supporting table folder: C:\Users\wenlu\OneDrive\books_coding\Python_Projects\hydrovoltaic-ml\results\supporting_materials\tables
Generated: 2026-05-26 14:21:04


In [2]:
def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")
    return path


def read_csv(path):
    path = require_file(path)
    return pd.read_csv(path, encoding="utf-8-sig", float_precision="round_trip")


def pick_col(df, candidates):
    """
    Return the first matching column name.
    """
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_text(x):
    return str(x).strip().replace("_", " ")


def write_table(df, filename):
    """
    Save one SI table CSV.
    """
    out_path = TABLE_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.17g")
    print("Saved:", out_path.name, df.shape)
    return out_path


def safe_sheet_name(name):
    """
    Excel sheet names must be <=31 characters.
    """
    return name[:31]

In [3]:
# =========================
# Figure 1 / dataset overview
# =========================
fig1_dir = RESULTS_DIR / "figure1_dataset_overview"

fig1_mech = read_csv(fig1_dir / "fig1C_mechanism_counts.csv")
fig1_structure = read_csv(fig1_dir / "fig1C_structure_counts.csv")
fig1_ion = read_csv(fig1_dir / "fig1C_ion_type_counts.csv")
fig1_completeness = read_csv(fig1_dir / "fig1D_reporting_completeness.csv")

# =========================
# Figure 3 / model comparison
# =========================
fig3_dir = RESULTS_DIR / "figure3_model_comparison"

# Preserve the validated Figure 3 numeric strings exactly in Tables S4-S6.
fig3A = pd.read_csv(fig3_dir / "fig3A_tuned_model_comparison.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
fig3B = pd.read_csv(fig3_dir / "fig3B_modelA_modelB_descriptor_augmentation.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
fig3C = pd.read_csv(fig3_dir / "fig3C_feature_block_ablation.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
fig3D = pd.read_csv(fig3_dir / "fig3D_SHAP_descriptor_importance.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)

# =========================
# Canonical Table S7(C-D) resistance analyses
# =========================
# Read the corrected Notebook 11 outputs directly. Do not route Table S7(D)
# through the historical Figure 4 hard-coded CSV.
s7c_canonical = read_csv(RESULTS_DIR / "11_R_regime_performance_statistics.csv")
s7d_canonical = read_csv(RESULTS_DIR / "11_physics_R_descriptor_model_comparison.csv")

# =========================
# Figure 5 / R origin
# =========================
fig5_dir = RESULTS_DIR / "figure5_R_origin"

fig5_raw = read_csv(fig5_dir / "fig5_raw_observed_R_data.csv")
fig5A_summary = read_csv(fig5_dir / "fig5A_inorganic_electrolyte_summary.csv")
fig5B_summary = read_csv(fig5_dir / "fig5B_ion_type_summary.csv")
fig5C_summary = read_csv(fig5_dir / "fig5C_structure_class_summary.csv")
fig5_tests = read_csv(fig5_dir / "fig5_statistical_tests.csv")

# =========================
# Figure 6 / explicit descriptor and virtual design
# =========================
fig6_dir = RESULTS_DIR / "figure6_virtual_design"

fig6A_coeff = read_csv(fig6_dir / "fig6A_linear_descriptor_coefficients_plot.csv")
fig6B_model = read_csv(fig6_dir / "fig6B_linear_vs_polynomial_model_comparison.csv")
fig6D_candidates = read_csv(fig6_dir / "fig6D_top_virtual_design_candidates.csv")

print("Loaded all source CSV files.")

Loaded all source CSV files.


In [4]:
table_s1 = pd.DataFrame([
    {
        "column_name": "paper_doi",
        "definition": "Digital object identifier or publication identifier for grouping records by source study.",
        "unit": "",
        "data_type": "string",
        "allowed_values": "free text",
        "missing_handling": "records without source identifier should be checked manually",
        "notes": "Used for paper-level validation where available."
    },
    {
        "column_name": "voc_V",
        "definition": "Open-circuit voltage reported for the device.",
        "unit": "V",
        "data_type": "numeric",
        "allowed_values": "> 0",
        "missing_handling": "required for final modeled dataset",
        "notes": "Used to construct estimated power density."
    },
    {
        "column_name": "jsc_uA_cm2",
        "definition": "Short-circuit current density reported for the device.",
        "unit": "μA cm−2",
        "data_type": "numeric",
        "allowed_values": "> 0",
        "missing_handling": "required for final modeled dataset",
        "notes": "Used to construct estimated power density."
    },
    {
        "column_name": "estimated_power_density_uW_cm2",
        "definition": "Estimated power density calculated from open-circuit voltage and short-circuit current density.",
        "unit": "μW cm−2",
        "data_type": "numeric",
        "allowed_values": "> 0",
        "missing_handling": "computed from Voc and Jsc",
        "notes": "Defined as P_est ≈ Voc × Jsc / 4 under a linear I–V approximation."
    },
    {
        "column_name": "log_estimated_power_density",
        "definition": "Base-10 logarithm of estimated power density.",
        "unit": "log10(μW cm−2)",
        "data_type": "numeric",
        "allowed_values": "real number",
        "missing_handling": "computed when estimated_power_density_uW_cm2 > 0",
        "notes": "Main modeling target."
    },
    {
        "column_name": "power_density_uW_cm2",
        "definition": "Reported power density from the original study, when available.",
        "unit": "μW cm−2",
        "data_type": "numeric",
        "allowed_values": ">= 0",
        "missing_handling": "not required for final modeled dataset",
        "notes": "Used as a reported metric but not as the primary target because reporting is incomplete."
    },
    {
        "column_name": "internal_resistance_Mohm",
        "definition": "Internal resistance reported or estimated for the device.",
        "unit": "MΩ",
        "data_type": "numeric",
        "allowed_values": "> 0",
        "missing_handling": "available subset used for resistance analysis",
        "notes": "Key transport descriptor."
    },
    {
        "column_name": "log_internal_resistance_Mohm",
        "definition": "Base-10 logarithm of internal resistance in MΩ.",
        "unit": "log10(MΩ)",
        "data_type": "numeric",
        "allowed_values": "real number",
        "missing_handling": "computed when internal_resistance_Mohm > 0",
        "notes": "Used as log(R) in the manuscript."
    },
    {
        "column_name": "mechanism_simple",
        "definition": "Simplified literature-reported mechanism label.",
        "unit": "",
        "data_type": "categorical",
        "allowed_values": "ion_gradient; streaming",
        "missing_handling": "records with ambiguous mechanism were harmonized manually",
        "notes": "Treated as reported label, not as ground-truth mechanism."
    },
    {
        "column_name": "structure_class",
        "definition": "Device structural class.",
        "unit": "",
        "data_type": "categorical",
        "allowed_values": "porous; film; hydrogel; hydrogel + porous; hydrogel + film",
        "missing_handling": "harmonized manually from literature description",
        "notes": "Used in descriptor analysis and structure-related comparisons."
    },
    {
        "column_name": "ion_type",
        "definition": "Primary mobile ion environment or dominant ion category.",
        "unit": "",
        "data_type": "categorical",
        "allowed_values": "proton; other_cation; anion",
        "missing_handling": "harmonized manually",
        "notes": "Encodes broad ionic environment rather than universal individual-ion ranking."
    },
    {
        "column_name": "inorganic_electrolyte_present",
        "definition": "Whether an inorganic electrolyte or salt descriptor is present.",
        "unit": "",
        "data_type": "binary/categorical",
        "allowed_values": "0/1 or equivalent category",
        "missing_handling": "harmonized manually",
        "notes": "Used to analyze the role of electrolyte environment in controlling R."
    },
    {
        "column_name": "material_class",
        "definition": "Broad material class of the active hydrovoltaic layer.",
        "unit": "",
        "data_type": "categorical",
        "allowed_values": "carbon; polymer; hydrogel; biomass; metal oxide; composite; semiconductor; others",
        "missing_handling": "harmonized manually when available",
        "notes": "Used as a descriptor block in modeling."
    },
    {
        "column_name": "R_regime",
        "definition": "Resistance-regime class based on log(R).",
        "unit": "",
        "data_type": "categorical",
        "allowed_values": "low_R; high_R",
        "missing_handling": "computed only when log(R) is available",
        "notes": "low_R is defined as log(R) ≤ −1; high_R is defined as log(R) > −1."
    },
])

write_table(table_s1, "Table_S1_data_dictionary_descriptor_encoding.csv")
display(table_s1)

Saved: Table_S1_data_dictionary_descriptor_encoding.csv (14, 7)


,column_name,definition,unit,data_type,allowed_values,missing_handling,notes
0,paper_doi,Digital object identifier or publication ident...,,string,free text,records without source identifier should be ch...,Used for paper-level validation where available.
1,voc_V,Open-circuit voltage reported for the device.,V,numeric,> 0,required for final modeled dataset,Used to construct estimated power density.
2,jsc_uA_cm2,Short-circuit current density reported for the...,μA cm−2,numeric,> 0,required for final modeled dataset,Used to construct estimated power density.
3,estimated_power_density_uW_cm2,Estimated power density calculated from open-c...,μW cm−2,numeric,> 0,computed from Voc and Jsc,Defined as P_est ≈ Voc × Jsc / 4 under a linea...
4,log_estimated_power_density,Base-10 logarithm of estimated power density.,log10(μW cm−2),numeric,real number,computed when estimated_power_density_uW_cm2 > 0,Main modeling target.
5,power_density_uW_cm2,Reported power density from the original study...,μW cm−2,numeric,>= 0,not required for final modeled dataset,Used as a reported metric but not as the prima...
6,internal_resistance_Mohm,Internal resistance reported or estimated for ...,MΩ,numeric,> 0,available subset used for resistance analysis,Key transport descriptor.
7,log_internal_resistance_Mohm,Base-10 logarithm of internal resistance in MΩ.,log10(MΩ),numeric,real number,computed when internal_resistance_Mohm > 0,Used as log(R) in the manuscript.
8,mechanism_simple,Simplified literature-reported mechanism label.,,categorical,ion_gradient; streaming,records with ambiguous mechanism were harmoniz...,"Treated as reported label, not as ground-truth..."
9,structure_class,Device structural class.,,categorical,porous; film; hydrogel; hydrogel + porous; hyd...,harmonized manually from literature description,Used in descriptor analysis and structure-rela...


In [5]:
table_s2 = pd.DataFrame([
    {
        "device_or_system_type": "Non-galvanic hydrovoltaic devices",
        "included_or_excluded": "Included",
        "reason": "Devices are intended to generate electricity from water, moisture, evaporation, or related hydrovoltaic processes without being dominated by galvanic electrode reactions.",
        "notes": "This is the target scope of the dataset."
    },
    {
        "device_or_system_type": "Droplet triboelectric / contact electrification systems",
        "included_or_excluded": "Excluded",
        "reason": "The dominant mechanism is contact electrification or triboelectric charge separation rather than the hydrovoltaic process considered here.",
        "notes": "Excluded to avoid mixing fundamentally different device physics."
    },
    {
        "device_or_system_type": "Salinity-gradient osmotic membrane power systems",
        "included_or_excluded": "Excluded",
        "reason": "These systems are primarily driven by salinity-gradient or osmotic membrane power mechanisms rather than the hydrovoltaic device class targeted here.",
        "notes": "Excluded from the curated non-galvanic hydrovoltaic dataset."
    },
    {
        "device_or_system_type": "Active-electrode redox-dominant systems",
        "included_or_excluded": "Excluded",
        "reason": "The output can be dominated by electrode redox reactions rather than transport-limited hydrovoltaic conversion.",
        "notes": "Examples include systems where electrode consumption or active electrochemistry is central to the reported output."
    },
    {
        "device_or_system_type": "Galvanic or battery-like configurations",
        "included_or_excluded": "Excluded",
        "reason": "These configurations behave more like electrochemical cells, making their performance non-comparable with non-galvanic hydrovoltaic devices.",
        "notes": "Excluded to maintain physical consistency in model interpretation."
    },
])

write_table(table_s2, "Table_S2_scope_exclusion_criteria.csv")
display(table_s2)

Saved: Table_S2_scope_exclusion_criteria.csv (5, 4)


,device_or_system_type,included_or_excluded,reason,notes
0,Non-galvanic hydrovoltaic devices,Included,Devices are intended to generate electricity f...,This is the target scope of the dataset.
1,Droplet triboelectric / contact electrificatio...,Excluded,The dominant mechanism is contact electrificat...,Excluded to avoid mixing fundamentally differe...
2,Salinity-gradient osmotic membrane power systems,Excluded,These systems are primarily driven by salinity...,Excluded from the curated non-galvanic hydrovo...
3,Active-electrode redox-dominant systems,Excluded,The output can be dominated by electrode redox...,Examples include systems where electrode consu...
4,Galvanic or battery-like configurations,Excluded,These configurations behave more like electroc...,Excluded to maintain physical consistency in m...


In [6]:
# Part A: reporting completeness
table_s3a = fig1_completeness.copy()

# Clean column order if expected columns exist
preferred_cols = [
    "variable_group",
    "columns_used",
    "n_available",
    "n_total",
    "availability_percent",
]

table_s3a = table_s3a[[c for c in preferred_cols if c in table_s3a.columns]]

write_table(table_s3a, "Table_S3A_reporting_completeness.csv")
display(table_s3a)

Saved: Table_S3A_reporting_completeness.csv (8, 5)


,variable_group,columns_used,n_available,n_total,availability_percent
0,Voc,voc_V,159,159,100.000000
1,Jsc,jsc_uA_cm2,159,159,100.000000
2,Reported power density,power_density_uW_cm2,108,159,67.924528
3,Internal resistance,internal_resistance_Mohm,112,159,70.440252
4,Structure,structure_class,159,159,100.000000
5,Ion type,ion_type,159,159,100.000000
6,Electrolyte descriptor,inorganic_electrolyte_present OR inorganic_ele...,159,159,100.000000
7,Mechanism label,mechanism_simple,159,159,100.000000


In [7]:
# Part B: category counts
def standardize_count_table(df, category_type, category_col):
    out = df.copy()
    out = out.rename(columns={category_col: "category"})
    out["category_type"] = category_type
    
    # Standard column order
    cols = ["category_type", "category", "count", "percent"]
    return out[[c for c in cols if c in out.columns]]


table_s3b = pd.concat(
    [
        standardize_count_table(fig1_mech, "mechanism_label", "mechanism_simple"),
        standardize_count_table(fig1_structure, "structure_class", "structure_class"),
        standardize_count_table(fig1_ion, "ion_type", "ion_type"),
    ],
    ignore_index=True
)

write_table(table_s3b, "Table_S3B_descriptor_category_counts.csv")
display(table_s3b)

Saved: Table_S3B_descriptor_category_counts.csv (10, 4)


,category_type,category,count,percent
0,mechanism_label,ion_gradient,107,67.295597
1,mechanism_label,streaming,52,32.704403
2,structure_class,porous,118,74.213836
3,structure_class,film,22,13.836478
4,structure_class,hydrogel,14,8.805031
5,structure_class,hydrogel + porous,4,2.515723
6,structure_class,hydrogel + film,1,0.628931
7,ion_type,proton,96,60.377358
8,ion_type,other_cation,54,33.962264
9,ion_type,anion,9,5.660377


In [8]:
# Table S4: validated R1C2 Figure 3A model-family comparison.
table_s4 = fig3A.rename(columns={
    "cv_r2_mean": "R2_CV_mean",
    "cv_r2_std": "R2_CV_std",
}).copy()
table_s4["analysis_population"] = "common n=112 observed-R records"
table_s4["resistance_handling"] = "no imputation"
table_s4["validation_protocol"] = table_s4["cv_n_splits"].map(lambda n: f"StratifiedShuffleSplit, {n} splits, seed 42")
table_s4["preprocessing"] = "training-fold categorical most-frequent imputation/one-hot encoding; numeric median imputation where applicable"
table_s4["selected_hyperparameters"] = "Validated R1C2 tuned settings; see results/revision/R1C2/analysis_metadata.json"

preferred_cols = [
    "model",
    "descriptor_set",
    "analysis_population",
    "resistance_handling",
    "validation_protocol",
    "preprocessing",
    "selected_hyperparameters",
    "n_rows",
    "R2_CV_mean",
    "R2_CV_std",
    "cv_split_fingerprint",
]

table_s4 = table_s4[[c for c in preferred_cols if c in table_s4.columns]]

write_table(table_s4, "Table_S4_model_hyperparameters_full_performance.csv")
display(table_s4)

Saved: Table_S4_model_hyperparameters_full_performance.csv (4, 5)


,model,preprocessing,selected_hyperparameters,R2_CV_mean,R2_CV_std
0,Elastic Net,one-hot encoding for categorical descriptors; ...,See model tuning notebook / to be filled from ...,0.228969,0.075275
1,SVR,one-hot encoding for categorical descriptors; ...,See model tuning notebook / to be filled from ...,0.213595,0.117491
2,Random Forest,one-hot encoding for categorical descriptors; ...,See model tuning notebook / to be filled from ...,0.252316,0.128247
3,XGBoost,one-hot encoding for categorical descriptors; ...,See model tuning notebook / to be filled from ...,0.254011,0.098608


In [9]:
# Part A: validated R1C2 descriptor augmentation on identical records/splits.
table_s5a = fig3B.rename(columns={
    "model_A": "Model_A_R2_CV",
    "model_A_std": "Model_A_R2_CV_std",
    "model_B": "Model_B_plus_logR_R2_CV",
    "model_B_std": "Model_B_plus_logR_R2_CV_std",
    "delta_B_minus_A": "Delta_R2_CV",
    "delta_B_minus_A_std": "Delta_R2_CV_std",
}).copy()
table_s5a["analysis_population"] = "common n=112 observed-R records; identical Model A/B splits"
table_s5a["resistance_handling"] = "no imputation"
table_s5a["table_part"] = "A. Descriptor augmentation"

preferred_cols = [
    "table_part",
    "model",
    "analysis_population",
    "resistance_handling",
    "n_rows",
    "cv_n_splits",
    "Model_A_R2_CV",
    "Model_A_R2_CV_std",
    "Model_B_plus_logR_R2_CV",
    "Model_B_plus_logR_R2_CV_std",
    "Delta_R2_CV",
    "Delta_R2_CV_std",
    "cv_split_fingerprint",
]

table_s5a = table_s5a[[c for c in preferred_cols if c in table_s5a.columns]]

write_table(table_s5a, "Table_S5A_descriptor_augmentation.csv")
display(table_s5a)

Saved: Table_S5A_descriptor_augmentation.csv (4, 5)


,table_part,model,Model_A_R2_CV,Model_B_plus_logR_R2_CV,Delta_R2_CV
0,A. Descriptor augmentation,Elastic Net,0.198247,0.266472,0.068225
1,A. Descriptor augmentation,Random Forest,0.106106,0.185757,0.079651
2,A. Descriptor augmentation,XGBoost,-0.061187,0.006350,0.067537
3,A. Descriptor augmentation,SVR,-0.016650,0.000526,0.017176


In [10]:
# Part B: validated R1C2 feature-block ablation on the common observed-R subset.
table_s5b = fig3C.rename(columns={
    "r2_mean": "R2_CV_after_removal",
    "r2_std": "R2_CV_after_removal_std",
    "delta_vs_baseline": "Delta_R2_CV_vs_full_model",
}).copy()
table_s5b["analysis_population"] = "common n=112 observed-R records"
table_s5b["resistance_handling"] = "no imputation"

table_s5b["table_part"] = "B. Feature-block ablation"

# Add interpretation
if "Delta_R2_CV_vs_full_model" in table_s5b.columns:
    table_s5b["interpretation"] = np.where(
        pd.to_numeric(table_s5b["Delta_R2_CV_vs_full_model"]) < -0.005,
        "Removing this block reduced model performance",
        "Removing this block caused little loss or slight improvement"
    )

preferred_cols = [
    "table_part",
    "removed_block",
    "analysis_population",
    "resistance_handling",
    "n_rows",
    "cv_n_splits",
    "full_model_r2",
    "R2_CV_after_removal",
    "R2_CV_after_removal_std",
    "Delta_R2_CV_vs_full_model",
    "interpretation",
    "cv_split_fingerprint",
]

table_s5b = table_s5b[[c for c in preferred_cols if c in table_s5b.columns]]

write_table(table_s5b, "Table_S5B_feature_block_ablation.csv")
display(table_s5b)

Saved: Table_S5B_feature_block_ablation.csv (5, 4)


,table_part,removed_block,Delta_R2_CV_vs_full_model,interpretation
0,B. Feature-block ablation,Mechanism labels,0.014126,Removing this block caused little loss or slig...
1,B. Feature-block ablation,Material class,-0.002009,Removing this block caused little loss or slig...
2,B. Feature-block ablation,Ion type,-0.002124,Removing this block caused little loss or slig...
3,B. Feature-block ablation,Internal resistance,-0.063061,Removing this block reduced model performance
4,B. Feature-block ablation,Structure,-0.078258,Removing this block reduced model performance


In [11]:
# Table S6: validated R1C2 full SHAP ranking on the observed-R population.
table_s6 = fig3D.rename(columns={
    "feature": "descriptor",
    "mean_abs_shap": "mean_abs_SHAP",
}).copy()

if "mean_abs_SHAP" in table_s6.columns:
    table_s6 = table_s6.sort_values("mean_abs_SHAP", ascending=False, key=pd.to_numeric).reset_index(drop=True)
    table_s6["rank"] = np.arange(1, len(table_s6) + 1)

preferred_cols = [
    "rank", "descriptor", "encoded_feature", "mean_abs_SHAP",
    "n_rows_observed_R", "holdout_random_state",
    "analysis_subset", "resistance_imputation",
]
table_s6 = table_s6[[c for c in preferred_cols if c in table_s6.columns]]

write_table(table_s6, "Table_S6_SHAP_descriptor_importance.csv")
display(table_s6)

Saved: Table_S6_SHAP_descriptor_importance.csv (10, 3)


,rank,descriptor,mean_abs_SHAP
0,1,log(R),0.452154
1,2,Other cation,0.132476
2,3,Inorganic electrolyte,0.097737
3,4,Porous structure,0.095245
4,5,Film structure,0.066256
5,6,Ion-gradient label,0.063009
6,7,Streaming label,0.050077
7,8,Metal electrode,0.035731
8,9,Carboxyl/hydroxyl groups,0.027268
9,10,Semiconductor,0.024974


In [12]:
def add_comparison_label(df, comparison):
    out = df.copy()
    out["comparison"] = comparison
    return out


table_s7a = pd.concat(
    [
        add_comparison_label(fig5A_summary, "inorganic electrolyte condition"),
        add_comparison_label(fig5B_summary, "ion type"),
        add_comparison_label(fig5C_summary, "structure class"),
    ],
    ignore_index=True
)

# R1C6: the direct structure-level inference table contains only adequately
# represented structure classes; sparse hybrids remain descriptive taxonomy only.
structure_rows_s7 = table_s7a.loc[table_s7a["comparison"] == "structure class", "structure_class_clean"].dropna()
if not set(structure_rows_s7).issubset({"porous", "film", "hydrogel"}):
    raise ValueError("Table S7 structure inference contains a sparse hybrid category.")

# Put comparison first
cols = ["comparison"] + [c for c in table_s7a.columns if c != "comparison"]
table_s7a = table_s7a[cols]

write_table(table_s7a, "Table_S7A_internal_resistance_group_statistics.csv")
display(table_s7a)

Saved: Table_S7A_internal_resistance_group_statistics.csv (10, 10)


,comparison,inorganic_electrolyte_group,count,mean,median,std,min,max,ion_type_clean,structure_class_clean
0,inorganic electrolyte condition,without_inorganic_electrolyte,71,-0.234584,0.000000,1.351894,-3.301030,3.000000,NaN,NaN
1,inorganic electrolyte condition,with_inorganic_electrolyte,41,-1.426768,-1.657577,1.018760,-3.468521,0.903090,NaN,NaN
2,ion type,NaN,62,-0.236174,0.000000,1.353036,-3.301030,3.000000,proton,NaN
3,ion type,NaN,42,-1.403316,-1.638683,1.045740,-3.468521,0.903090,other_cation,NaN
4,ion type,NaN,8,-0.196365,-0.183620,1.405647,-2.000000,1.778151,anion,NaN
5,structure class,NaN,84,-0.429170,-0.227966,1.370990,-3.468521,3.000000,NaN,porous
6,structure class,NaN,13,-0.946162,-1.000000,1.172801,-3.301030,1.301030,NaN,film
7,structure class,NaN,10,-1.708592,-2.000000,0.915122,-3.000000,0.342423,NaN,hydrogel
8,structure class,NaN,4,-2.079690,-1.849485,0.635027,-3.000000,-1.619789,NaN,hydrogel_+_porous
9,structure class,NaN,1,-1.397940,-1.397940,NaN,-1.397940,-1.397940,NaN,hydrogel_+_film


In [13]:
# Statistical tests
table_s7b = fig5_tests.copy()

write_table(table_s7b, "Table_S7B_internal_resistance_statistical_tests.csv")
display(table_s7b)

Saved: Table_S7B_internal_resistance_statistical_tests.csv (3, 5)


,panel,comparison,test,statistic,p_value
0,Fig5A,inorganic electrolyte presence,Mann-Whitney U,2218.000000,0.000004
1,Fig5B,ion type,Kruskal-Wallis,20.407733,0.000037
2,Fig5C,structure class,Kruskal-Wallis,13.753155,0.008126


In [14]:
# Resistance-regime performance statistics from the canonical observed-R analysis
table_s7c = s7c_canonical.copy()
if table_s7c["count"].sum() != 112:
    raise AssertionError("Table S7(C) counts must sum to 112.")

write_table(table_s7c, "Table_S7C_resistance_regime_performance_statistics.csv")
display(table_s7c)

Saved: Table_S7C_resistance_regime_performance_statistics.csv (2, 8)


,R_regime,threshold,count,mean_log_P_est,median_log_P_est,std_log_P_est,mean_log_R,median_log_R
0,low_R,"log(R) = -1; low_R if log(R) <= -1, high_R if ...",55,0.413553,0.454845,0.957220,-1.807239,-1.721246
1,high_R,"log(R) = -1; low_R if log(R) <= -1, high_R if ...",57,-0.850091,-0.823906,1.243873,0.425354,0.176091


In [15]:
# Corrected observed-R descriptor-formulation comparison
model_order = ["baseline_logR", "centered_logR", "R_regime_descriptor"]
model_labels = {
    "baseline_logR": "Raw log(R)",
    "centered_logR": "Centered log(R)",
    "R_regime_descriptor": "R-regime descriptor",
}
table_s7d = (
    s7d_canonical
    .set_index("model")
    .reindex(model_order)
    .reset_index()
    .rename(columns={"model": "descriptor_model"})
)
if not (table_s7d["n_samples"] == 112).all():
    raise AssertionError("Every Table S7(D) formulation must use n=112.")
table_s7d.insert(1, "label", table_s7d["descriptor_model"].map(model_labels))
table_s7d = table_s7d[[
    "descriptor_model", "label", "n_samples",
    "cv_R2_mean", "cv_R2_std", "test_R2",
]]

write_table(table_s7d, "Table_S7D_R_descriptor_comparison.csv")
display(table_s7d)

Saved: Table_S7D_R_descriptor_comparison.csv (3, 5)


,descriptor_model,label,cv_R2_mean,cv_R2_std,test_R2
0,baseline_logR,Raw\nlog(R),0.251254,0.091204,0.428578
1,centered_logR,Centered\nlog(R),0.251254,0.091204,0.428578
2,R_regime_descriptor,R-regime\ndescriptor,0.296935,0.131584,0.518388


In [16]:
# Part A: R1C6 sparse-hybrid-excluded explicit descriptor coefficients
if fig6A_coeff["feature"].astype(str).str.contains(r"hydrogel_\+", regex=True).any():
    raise ValueError("Table S8A contains a sparse hybrid structure coefficient.")
table_s8a = fig6A_coeff.copy()

write_table(table_s8a, "Table_S8A_explicit_descriptor_model_coefficients.csv")
display(table_s8a)

Saved: Table_S8A_explicit_descriptor_model_coefficients.csv (10, 4)


,feature,coefficient,abs_coefficient,label
0,cat__structure_clean_film,0.999513,0.999513,Structure: film
1,cat__structure_clean_hydrogel_+_film,0.599472,0.599472,Structure: hydrogel + film
2,cat__ion_type_clean_other_cation,0.505642,0.505642,Ion: other cation
3,num__log_R,-0.469094,0.469094,log(R)
4,cat__structure_clean_porous,-0.295976,0.295976,Structure: porous
5,cat__ion_type_clean_anion,-0.273914,0.273914,Ion: anion
6,num__inorganic_electrolyte_present_clean,0.072311,0.072311,Inorganic electrolyte
7,cat__mechanism_clean_ion_gradient,0.069794,0.069794,Mechanism: ion gradient
8,cat__structure_clean_hydrogel,-0.065805,0.065805,Structure: hydrogel
9,cat__mechanism_clean_streaming,-0.022199,0.022199,Mechanism: streaming


In [17]:
# Part B: linear vs polynomial descriptor model comparison
table_s8b = fig6B_model.copy()

write_table(table_s8b, "Table_S8B_descriptor_model_comparison.csv")
display(table_s8b)

Saved: Table_S8B_descriptor_model_comparison.csv (2, 11)


,model,train_R2,test_R2,test_RMSE,test_MAE,cv_R2_mean,cv_R2_std,cv_RMSE_mean,cv_RMSE_std,cv_MAE_mean,cv_MAE_std
0,Linear Elastic Net descriptor,0.447677,0.341504,0.900899,0.684797,0.298469,0.133919,1.034126,0.095595,0.804468,0.068022
1,Polynomial Elastic Net descriptor,0.440545,0.280680,0.941588,0.751510,0.228305,0.062941,1.094372,0.129098,0.865327,0.102349


In [18]:
# Part C: R1C6 top virtual design candidates
if not set(fig6D_candidates["structure_clean"]).issubset({"porous", "film", "hydrogel"}):
    raise ValueError("Table S8C contains a sparse hybrid virtual candidate.")
table_s8c = fig6D_candidates.copy()

# Add rank if not already present
if "rank" not in table_s8c.columns:
    pred_col = pick_col(table_s8c, ["predicted_log_P_est", "predicted_log_estimated_power_density"])
    if pred_col is not None:
        table_s8c = table_s8c.sort_values(pred_col, ascending=False).reset_index(drop=True)
        table_s8c["rank"] = np.arange(1, len(table_s8c) + 1)

# Put rank first if available
if "rank" in table_s8c.columns:
    cols = ["rank"] + [c for c in table_s8c.columns if c != "rank"]
    table_s8c = table_s8c[cols]

write_table(table_s8c, "Table_S8C_virtual_design_candidates.csv")
display(table_s8c)

Saved: Table_S8C_virtual_design_candidates.csv (10, 7)


,rank,structure_clean,ion_type_clean,inorganic_electrolyte_present_clean,log_R,R_regime,predicted_log_P_est
0,1,film,other_cation,1,-2.0,low_R,1.794374
1,2,film,other_cation,0,-2.0,low_R,1.644268
2,3,film,other_cation,1,-1.0,low_R,1.448812
3,4,hydrogel_+_film,other_cation,1,-2.0,low_R,1.394333
4,5,film,other_cation,0,-1.0,low_R,1.298706
5,6,film,proton,1,-2.0,low_R,1.288732
6,7,hydrogel_+_film,other_cation,0,-2.0,low_R,1.244226
7,8,film,proton,0,-2.0,low_R,1.138626
8,9,film,other_cation,1,0.0,high_R,1.103251
9,10,hydrogel_+_film,other_cation,1,-1.0,low_R,1.048771


In [19]:
# Do not rebuild Supporting_Tables_S1_to_S8.xlsx here.
# The former export recreated every S1-S8 sheet, which is not a safe targeted
# operation for targeted revisions. The regenerated Table_S4-S6 and
# Table_S7C-S7D CSVs are the authoritative updated sources; apply a
# worksheet-targeted update to the
# existing workbook only when a tool that preserves unaffected sheets/styles is available.
xlsx_path = TABLE_DIR / "Supporting_Tables_S1_to_S8.xlsx"
print("Targeted CSV tables regenerated; existing workbook left unchanged to preserve unaffected sheets and styles.")
print("Workbook target for a later worksheet-targeted update:", xlsx_path)

Saved Excel workbook:
C:\Users\wenlu\OneDrive\books_coding\Python_Projects\hydrovoltaic-ml\results\supporting_materials\tables\Supporting_Tables_S1_to_S8.xlsx


In [20]:
print("Supporting tables generated:")
print("----------------------------")

for p in sorted(TABLE_DIR.glob("Table_S*.csv")):
    print(" -", p.name)

xlsx_files = list(TABLE_DIR.glob("Supporting_Tables_S1_to_S8.xlsx"))

if xlsx_files:
    print("\nExcel workbook:")
    print(" -", xlsx_files[0].name)

Supporting tables generated:
----------------------------
 - Table_S1_data_dictionary_descriptor_encoding.csv
 - Table_S2_scope_exclusion_criteria.csv
 - Table_S3A_reporting_completeness.csv
 - Table_S3B_descriptor_category_counts.csv
 - Table_S4_model_hyperparameters_full_performance.csv
 - Table_S5A_descriptor_augmentation.csv
 - Table_S5B_feature_block_ablation.csv
 - Table_S6_SHAP_descriptor_importance.csv
 - Table_S7A_internal_resistance_group_statistics.csv
 - Table_S7B_internal_resistance_statistical_tests.csv
 - Table_S7C_resistance_regime_performance_statistics.csv
 - Table_S7D_R_descriptor_comparison.csv
 - Table_S8A_explicit_descriptor_model_coefficients.csv
 - Table_S8B_descriptor_model_comparison.csv
 - Table_S8C_virtual_design_candidates.csv

Excel workbook:
 - Supporting_Tables_S1_to_S8.xlsx
